In [193]:
import pandas as pd
import gc

In [194]:
data = "../data/fines.csv"

In [195]:
df = pd.read_csv(data)

## **Итерации**

In [196]:
df_copy = df.copy()

### **Цикл с iloc**

In [197]:
def calculate_iloc(df):
    result = []
    for i in range(0, len(df)):
        res = df.iloc[i]["Fines"] / df.iloc[i]["Refund"] * df.iloc[i]["Year"]
        result.append(res)
    return result

In [198]:
%%timeit
calculate_iloc(df_copy)

29.6 ms ± 82.8 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### **iterrows()**

In [199]:
def calculate_iterrows(df):
    result = []
    for idx, row in df.iterrows():
        res = row["Fines"] / row["Refund"] * row["Year"]
        result.append(res)
    return result

In [200]:
%%timeit
calculate_iterrows(df_copy)

10.1 ms ± 153 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### **apply()**

In [201]:
def calculate_apply(df):
    return df.apply(lambda row: row["Fines"] / row["Refund"] * row["Year"], axis=1)

In [202]:
%%timeit
calculate_apply(df_copy)

3.01 ms ± 18.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### **Series**

In [203]:
def calculate_series(df):
    return df["Fines"] / df["Refund"] * df["Year"]

In [204]:
%%timeit
calculate_series(df_copy)

30.6 µs ± 388 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### **.values**

In [205]:
def calculate_values(df):
    return df["Fines"].values / df["Refund"].values * df["Year"].values

In [206]:
%%timeit
calculate_values(df_copy)

4.99 µs ± 68.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


### **DataFrame с новыми столбцами**

In [207]:
df["res_iloc"] = calculate_iloc(df)
df["res_iterrows"] = calculate_iterrows(df)
df["res_apply"] = calculate_apply(df)
df["res_series"] = calculate_series(df)
df["res_values"] = calculate_values(df)

In [208]:
df.head(10)

,CarNumber,Refund,Fines,Make,Model,Year,res_iloc,res_iterrows,res_apply,res_series,res_values
0,Y163O8161RUS,2,3200.000000,Ford,Focus,1989,3.182400e+06,3.182400e+06,3.182400e+06,3.182400e+06,3.182400e+06
1,E432XX77RUS,1,6500.000000,Toyota,Camry,1995,1.296750e+07,1.296750e+07,1.296750e+07,1.296750e+07,1.296750e+07
2,7184TT36RUS,1,2100.000000,Ford,Focus,1984,4.166400e+06,4.166400e+06,4.166400e+06,4.166400e+06,4.166400e+06
3,X582HE161RUS,2,2000.000000,Ford,Focus,2015,2.015000e+06,2.015000e+06,2.015000e+06,2.015000e+06,2.015000e+06
4,92918M178RUS,1,5700.000000,Ford,Focus,2014,1.147980e+07,1.147980e+07,1.147980e+07,1.147980e+07,1.147980e+07
5,H234YH197RUS,2,6000.000000,Ford,Focus,1990,5.970000e+06,5.970000e+06,5.970000e+06,5.970000e+06,5.970000e+06
6,E40577152RUS,1,8594.586466,Ford,Focus,1988,1.708604e+07,1.708604e+07,1.708604e+07,1.708604e+07,1.708604e+07
7,707987163RUS,2,2200.000000,Ford,Focus,2016,2.217600e+06,2.217600e+06,2.217600e+06,2.217600e+06,2.217600e+06
8,K330T8197RUS,2,8200.000000,Skoda,Octavia,2018,8.273800e+06,8.273800e+06,8.273800e+06,8.273800e+06,8.273800e+06
9,X786CO96RUS,1,8594.586466,Ford,Focus,2000,1.718917e+07,1.718917e+07,1.718917e+07,1.718917e+07,1.718917e+07


## **Индексация**

### **Строка для "O136HO197RUS"**

In [209]:
%%timeit
df[df["CarNumber"] == "O136HO197RUS"].head(1)

92.9 µs ± 789 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### **Строка для "O136HO197RUS" (CarNumber - индекс)**

In [210]:
df = df.set_index("CarNumber")

In [211]:
%%timeit
df.loc["O136HO197RUS"].head(1)

40.9 µs ± 606 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## **Понижение разрядности типов (downcasting)**

### **df.info(memory_usage='deep')**

In [212]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to V222VV197RUS
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Refund        930 non-null    int64  
 1   Fines         930 non-null    float64
 2   Make          930 non-null    object 
 3   Model         919 non-null    object 
 4   Year          930 non-null    int64  
 5   res_iloc      930 non-null    float64
 6   res_iterrows  930 non-null    float64
 7   res_apply     930 non-null    float64
 8   res_series    930 non-null    float64
 9   res_values    930 non-null    float64
dtypes: float64(6), int64(2), object(2)
memory usage: 265.1 KB


### **optimized_df**

In [213]:
optimized_df = df.copy()

### **Понижение разрядности типа с float64 до float32 для всех столбцов**

In [214]:
for col in optimized_df.select_dtypes(include=["float64"]).columns:
    optimized_df[col] = optimized_df[col].astype("float32")

### **Понижение разрядности типа с int64 до минимально возможного числового Dtype**

In [215]:
for col in optimized_df.select_dtypes(include=["int64"]).columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast="integer")

### **info(memory_usage='deep') для нового DataFrame**

In [216]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to V222VV197RUS
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Refund        930 non-null    int8   
 1   Fines         930 non-null    float32
 2   Make          930 non-null    object 
 3   Model         919 non-null    object 
 4   Year          930 non-null    int16  
 5   res_iloc      930 non-null    float32
 6   res_iterrows  930 non-null    float32
 7   res_apply     930 non-null    float32
 8   res_series    930 non-null    float32
 9   res_values    930 non-null    float32
dtypes: float32(6), int16(1), int8(1), object(2)
memory usage: 231.5 KB


## **Категории**

### **Преобразование столбцев типа object в category**

In [217]:
for col in optimized_df.select_dtypes(include="object").columns:
    optimized_df[col] = optimized_df[col].astype("category")

### **info(memory_usage='deep')**

In [218]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to V222VV197RUS
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Refund        930 non-null    int8    
 1   Fines         930 non-null    float32 
 2   Make          930 non-null    category
 3   Model         919 non-null    category
 4   Year          930 non-null    int16   
 5   res_iloc      930 non-null    float32 
 6   res_iterrows  930 non-null    float32 
 7   res_apply     930 non-null    float32 
 8   res_series    930 non-null    float32 
 9   res_values    930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 122.8 KB


## **Очистка памяти**

In [219]:
%reset_selective df
gc.collect()

0